# Interactive guide to HBR parameters

This notebook is a hands-on playground for building intuition about the
parameters of a Hierarchical Bayesian Regression (HBR) model in the PCNtoolkit,
with a focus on the **SHASHb** likelihood.

An HBR model is always a likelihood plus priors on the parameters of that
likelihood. For the SHASHb likelihood there are four parameters:

| parameter | role | analogous to |
|-----------|------|--------------|
| `mu`      | location (mean)            | mean of a Normal |
| `sigma`   | scale (spread)             | std of a Normal  |
| `epsilon` | **skewness**  | — |
| `delta`   | **kurtosis** | — |

Each parameter is configured with `make_prior`, and can be:
- a **constant prior**, or **linear** in the covariates (with a basis function),
- **fixed** across batches or have a **random effect** per batch,
- passed through a **mapping** (e.g. `softplus`) to keep it positive.

In [17]:
import warnings

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown

# Reuse the toolkit's own SHASH math so this playground matches the model.
from pcntoolkit.math_functions.shash import S, S_inv, m1m2
from pcntoolkit import BsplineBasisFunction

warnings.simplefilter("ignore")
%matplotlib inline
plt.rcParams["figure.figsize"] = (9, 4.5)

## Helper functions

These two functions are the engine behind every plot below:

- `shashb_pdf` — the shape of the bell curve (how likely each value is).
- `shashb_centiles` — the lines on that curve (e.g. the 5th / 50th / 95th percentiles), the kind a growth chart would show.

In [ ]:
CONST2 = -0.5 * np.log(2 * np.pi)  # normal log-normalising constant


def shashb_pdf(y, mu, sigma, epsilon, delta):
    # Probability density of a SHASHb(mu, sigma, epsilon, delta) variable.
    # m1m2 returns (mean, raw second moment); variance = raw_second - mean^2.
    mean, raw_second = m1m2(epsilon, delta)
    true_sigma = np.sqrt(raw_second - mean ** 2)
    # Map y back to the standardised SHASH domain.
    x = ((y - mu) / sigma) * true_sigma + mean
    Sx = S(x, epsilon, delta)
    log_base = (
        CONST2
        + np.log(delta)
        + 0.5 * np.log1p(Sx ** 2)
        - 0.5 * np.log1p(x ** 2)
        - 0.5 * Sx ** 2
    )
    # Change-of-variables Jacobian for the (y -> x) affine map.
    return np.exp(log_base) * true_sigma / sigma


def shashb_centiles(centiles, mu, sigma, epsilon, delta):
    # Quantiles (centiles) of a SHASHb(mu, sigma, epsilon, delta) variable.
    z = norm.ppf(np.asarray(centiles, dtype=float))  # standard-normal quantiles
    mean, raw_second = m1m2(epsilon, delta)
    true_sigma = np.sqrt(raw_second - mean ** 2)
    x = S_inv(z, epsilon, delta)                      # base SHASH quantiles
    return ((x - mean) / true_sigma) * sigma + mu


# Sanity check: the density must integrate to ~1.
_grid = np.linspace(-15, 15, 20001)
_area = np.trapz(shashb_pdf(_grid, 0.0, 1.0, 0.8, 0.9), _grid)
print(f"pdf integrates to {_area:.4f}")

pdf integrates to 1.0000 (should be ~1.0)


## Section A — the likelihood shape: `epsilon` and `delta`

`mu` and `sigma` shift and scale the distribution, just like a Normal. The two
new parameters reshape it:

- **`epsilon` (skewness).** `epsilon = 0` is symmetric. Positive `epsilon` skews
  the distribution to the **right** (long right tail), negative to the left.
- **`delta` (tail weight).** `delta = 1` recovers normal-like tails. `delta < 1`
  gives **heavier** tails and a sharper peak; `delta > 1` gives **lighter** tails.

The dashed lines mark the 5th, 50th and 95th centiles — these are exactly the
centile curves the model would draw.

In [ ]:
@interact(
    epsilon=FloatSlider(value=0.0, min=-2.0, max=2.0, step=0.1,
                        description="epsilon"),
    delta=FloatSlider(value=1.0, min=0.3, max=2.5, step=0.05,
                      description="delta"),
    mu=FloatSlider(value=0.0, min=-3.0, max=3.0, step=0.1, description="mu"),
    sigma=FloatSlider(value=1.0, min=0.3, max=3.0, step=0.1, description="sigma"),
)
def _plot_shape(epsilon, delta, mu, sigma):
    y = np.linspace(-8, 8, 1000)
    pdf = shashb_pdf(y, mu, sigma, epsilon, delta)
    # A reference standard Normal with the same mu, sigma.
    ref = norm.pdf(y, mu, sigma)

    fig, ax = plt.subplots()
    ax.plot(y, pdf, lw=2.5, label="SHASHb")
    ax.plot(y, ref, lw=1.5, ls=":", color="grey", label="Normal(mu, sigma)")
    for c, ls in zip([0.05, 0.5, 0.95], ["--", "-", "--"]):
        q = shashb_centiles([c], mu, sigma, epsilon, delta)[0]
        ax.axvline(q, color="crimson", ls=ls, lw=1, alpha=0.7)
    ax.set_xlim(-8, 8)
    ax.set_ylim(bottom=0)
    ax.set_xlabel("y")
    ax.set_ylabel("density")
    ax.set_title("SHASHb density (red lines = 5th/50th/95th centiles)")
    ax.legend(loc="upper right")
    plt.show()

interactive(children=(FloatSlider(value=0.0, description='epsilon', max=2.0, min=-2.0), FloatSlider(value=1.0,…

## Section B — keeping parameters positive: the `softplus` mapping

`sigma` and `delta` must be **strictly positive**. Instead of constraining the
prior, the toolkit samples an unconstrained value and pushes it through a
`softplus` mapping. The mapping is parameterised by `(a, b, c)` and applied as:

```
f_abc(x) = softplus((x - a) / b) * b + c
```

- **`a`** — horizontal shift
- **`b`** — scale / smoothness (larger `b` = gentler, more linear)
- **`c`** — vertical shift (a **floor** on the output)

The vertical shift `c` matters for `delta`: the SHASH moments become numerically
unstable as `delta -> 0`, so configuring `delta` with a small floor
(e.g. `mapping_params=(0.0, 3.0, 0.6)`) keeps it safely away from that region.

In [ ]:
def softplus_mapping(x, a, b, c):
    # Affine-parameterised softplus, matching Prior.apply_mapping.
    return np.log1p(np.exp((x - a) / b)) * b + c


@interact(
    a=FloatSlider(value=0.0, min=-4.0, max=4.0, step=0.1, description="a (shift)"),
    b=FloatSlider(value=3.0, min=0.5, max=6.0, step=0.1, description="b (scale)"),
    c=FloatSlider(value=0.0, min=0.0, max=2.0, step=0.1, description="c (floor)"),
)
def _plot_softplus(a, b, c):
    x = np.linspace(-8, 8, 400)
    fig, ax = plt.subplots()
    ax.plot(x, softplus_mapping(x, 0.0, 1.0, 0.0), ls=":", color="grey",
            label="softplus (a=0, b=1, c=0)")
    ax.plot(x, softplus_mapping(x, a, b, c), lw=2.5,
            label=f"softplus (a={a}, b={b}, c={c})")
    ax.axhline(c, color="crimson", ls="--", lw=1, alpha=0.7,
               label=f"floor c={c}")
    ax.set_xlabel("unconstrained sample x")
    ax.set_ylabel("mapped (positive) value")
    ax.set_title("Effect of the softplus mapping parameters")
    ax.legend(loc="upper left")
    plt.show()

interactive(children=(FloatSlider(value=0.0, description='a (shift)', max=4.0, min=-4.0), FloatSlider(value=3.…

## Section C — priors and the prior predictive

Before fitting, each parameter has a **prior** describing what values are
plausible. For parameters that are *linear in the covariates* (like `mu` and
`sigma` here), the prior is placed on the **slope** and **intercept**
coefficients, and a **basis function** controls how flexible the curve can be.

### C.1 — choosing a prior distribution

Pick a distribution and see the implied density. This is what `dist_name` and
`dist_params` in `make_prior` control.

In [21]:
def sample_prior(dist_name, p0, p1, n=200000, seed=0):
    rng = np.random.default_rng(seed)
    if dist_name == "Normal":
        return rng.normal(p0, p1, n)
    if dist_name == "HalfNormal":
        return np.abs(rng.normal(0.0, p1, n))
    if dist_name == "LogNormal":
        return rng.lognormal(p0, p1, n)
    if dist_name == "Gamma":          # (alpha, beta) shape-rate, PyMC order
        return rng.gamma(max(p0, 1e-3), 1.0 / max(p1, 1e-3), n)
    if dist_name == "Uniform":
        lo, hi = min(p0, p1), max(p0, p1)
        return rng.uniform(lo, hi + 1e-9, n)
    raise ValueError(dist_name)


@interact(
    dist_name=Dropdown(
        options=["Normal", "HalfNormal", "LogNormal", "Gamma", "Uniform"],
        value="Normal", description="dist"),
    p0=FloatSlider(value=0.0, min=-3.0, max=5.0, step=0.1, description="param 0"),
    p1=FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1, description="param 1"),
)
def _plot_prior(dist_name, p0, p1):
    samples = sample_prior(dist_name, p0, p1)
    fig, ax = plt.subplots()
    ax.hist(samples, bins=200, density=True, color="steelblue", alpha=0.8)
    ax.set_xlim(np.percentile(samples, 0.5), np.percentile(samples, 99.5))
    ax.set_xlabel("parameter value")
    ax.set_ylabel("prior density")
    ax.set_title(f"{dist_name}(p0={p0}, p1={p1}) prior")
    plt.show()

interactive(children=(Dropdown(description='dist', options=('Normal', 'HalfNormal', 'LogNormal', 'Gamma', 'Uni…

### C.2 — the basis function: `nknots` and `degree`

A `BsplineBasisFunction` expands the covariate (age) into several smooth basis
columns. A linear `mu` is then a weighted sum of these columns plus an
intercept. More knots / higher degree = a more flexible (wigglier) curve.

The left panel shows the basis columns; the right shows a few `mu(age)` curves
drawn from the prior (slope coefficients ~ `Normal(0, slope_sd)`). This is the
**prior predictive** for the mean — what the model believes *before* seeing data.

In [ ]:
AGE = np.linspace(5, 85, 200)


def make_basis(nknots, degree):
    bf = BsplineBasisFunction(basis_column=0, nknots=nknots, degree=degree)
    x = AGE.reshape(-1, 1)
    bf.fit(x)
    return np.asarray(bf.transform(x))


@interact(
    nknots=IntSlider(value=5, min=3, max=12, step=1, description="nknots"),
    degree=IntSlider(value=3, min=1, max=5, step=1, description="degree"),
    slope_sd=FloatSlider(value=2.0, min=0.2, max=8.0, step=0.2,
                         description="slope_sd"),
    n_draws=IntSlider(value=8, min=1, max=30, step=1, description="n draws"),
)
def _plot_basis(nknots, degree, slope_sd, n_draws):
    phi = make_basis(nknots, degree)          # (n_age, k)
    rng = np.random.default_rng(0)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
    axes[0].plot(AGE, phi)
    axes[0].set_title(f"B-spline basis ({phi.shape[1]} columns)")
    axes[0].set_xlabel("age")
    for _ in range(n_draws):
        w = rng.normal(0.0, slope_sd, phi.shape[1])
        axes[1].plot(AGE, phi @ w, alpha=0.7)
    axes[1].set_title("mu(age) curves drawn from the prior")
    axes[1].set_xlabel("age")
    plt.tight_layout()
    plt.show()

interactive(children=(IntSlider(value=5, description='nknots', max=12, min=3), IntSlider(value=3, description=…

### C.3 — putting it together: a prior-predictive centile fan

Now combine everything: a `mu(age)` curve, a positive heteroskedastic
`sigma(age)` (via softplus), and the SHASH shape parameters `epsilon` and
`delta`. The shaded band is the 5th–95th centile fan the model would predict
*a priori*. Notice how `epsilon` makes the fan **asymmetric** around the median.

In [ ]:
@interact(
    nknots=IntSlider(value=5, min=3, max=10, step=1, description="nknots"),
    degree=IntSlider(value=3, min=1, max=5, step=1, description="degree"),
    slope_sd=FloatSlider(value=2.0, min=0.2, max=6.0, step=0.2,
                         description="mu slope_sd"),
    sigma_level=FloatSlider(value=1.0, min=0.2, max=3.0, step=0.1,
                            description="sigma level"),
    epsilon=FloatSlider(value=0.5, min=-1.5, max=1.5, step=0.1,
                        description="epsilon"),
    delta=FloatSlider(value=1.0, min=0.4, max=2.0, step=0.05,
                      description="delta"),
    seed=IntSlider(value=0, min=0, max=20, step=1, description="seed"),
)
def _plot_fan(nknots, degree, slope_sd, sigma_level, epsilon, delta, seed):
    phi = make_basis(nknots, degree)
    rng = np.random.default_rng(seed)
    w = rng.normal(0.0, slope_sd, phi.shape[1])
    mu_age = phi @ w
    # Heteroskedastic, strictly-positive sigma via softplus over age.
    w_s = rng.normal(0.0, 0.6, phi.shape[1])
    sigma_age = softplus_mapping(phi @ w_s, 0.0, 1.0, 0.0) * sigma_level + 0.2

    cents = [0.05, 0.25, 0.5, 0.75, 0.95]
    curves = np.array([
        [shashb_centiles([c], m, s, epsilon, delta)[0]
         for c in cents]
        for m, s in zip(mu_age, sigma_age)
    ])  # (n_age, n_cent)

    fig, ax = plt.subplots()
    ax.fill_between(AGE, curves[:, 0], curves[:, -1], alpha=0.2,
                    color="steelblue", label="5th-95th centile")
    ax.fill_between(AGE, curves[:, 1], curves[:, -2], alpha=0.3,
                    color="steelblue", label="25th-75th centile")
    ax.plot(AGE, curves[:, 2], color="navy", lw=2, label="median")
    ax.set_xlabel("age")
    ax.set_ylabel("response")
    ax.set_title("Prior-predictive centile fan (SHASHb)")
    ax.legend(loc="upper left")
    plt.show()

interactive(children=(IntSlider(value=5, description='nknots', max=10, min=3), IntSlider(value=3, description=…

## Section D — generate the `make_prior` configuration

Once the sliders give you a shape you like, this cell emits the matching
`make_prior` / `SHASHbLikelihood` code so you can paste it straight into a model.
Adjust the sliders and copy the printed snippet.

In [24]:
def build_config(mu_slope_sd, sigma_slope_sd,
                 nknots, degree, random_intercept):
    intercept = (
        "intercept=make_prior(\n"
        "        random=True,\n"
        "        mu=make_prior(dist_name=\"Normal\", dist_params=(0.0, 1.0)),\n"
        "        sigma=make_prior(dist_name=\"Normal\", dist_params=(0.0, 1.0),\n"
        "                         mapping=\"softplus\", mapping_params=(0.0, 3.0)),\n"
        "    ),"
        if random_intercept else
        "intercept=make_prior(dist_name=\"Normal\", dist_params=(0.0, 1.0)),"
    )
    return f'''mu = make_prior(
    linear=True,
    slope=make_prior(dist_name="Normal", dist_params=(0.0, {mu_slope_sd})),
    {intercept}
    basis_function=BsplineBasisFunction(basis_column=0, nknots={nknots}, degree={degree}),
)
sigma = make_prior(
    linear=True,
    slope=make_prior(dist_name="Normal", dist_params=(0.0, {sigma_slope_sd})),
    intercept=make_prior(dist_name="Normal", dist_params=(1.0, 1.0)),
    basis_function=BsplineBasisFunction(basis_column=0, nknots={nknots}, degree={degree}),
    mapping="softplus",
    mapping_params=(0.0, 3.0),
)
epsilon = make_prior(dist_name="Normal", dist_params=(0.0, 1.0))
delta = make_prior(
    dist_name="Normal",
    dist_params=(1.0, 1.0),
    mapping="softplus",
    mapping_params=(0.0, 3.0, 0.6),
)
likelihood = SHASHbLikelihood(mu, sigma, epsilon, delta)'''


@interact(
    mu_slope_sd=FloatSlider(value=10.0, min=1.0, max=20.0, step=1.0,
                            description="mu slope_sd"),
    sigma_slope_sd=FloatSlider(value=2.0, min=0.5, max=6.0, step=0.5,
                               description="sigma slope_sd"),
    nknots=IntSlider(value=5, min=3, max=10, step=1, description="nknots"),
    degree=IntSlider(value=3, min=1, max=5, step=1, description="degree"),
    random_intercept=Dropdown(options=[True, False], value=True,
                              description="random mu"),
)
def _show_config(mu_slope_sd, sigma_slope_sd, nknots, degree, random_intercept):
    print(build_config(mu_slope_sd, sigma_slope_sd,
                        nknots, degree, random_intercept))

interactive(children=(FloatSlider(value=10.0, description='mu slope_sd', max=20.0, min=1.0, step=1.0), FloatSl…

## Takeaways

- **`mu` / `sigma`** behave like a Normal's mean and std; **`epsilon`** controls
  skew and **`delta`** controls tail weight. Use a SHASHb likelihood when your
  feature is skewed or heavy-tailed (e.g. ventricle volumes).
- **Mappings** keep `sigma` and `delta` positive. The `delta` floor
  (`c` in `mapping_params`) avoids the numerically unstable `delta -> 0` region.
- **Priors + basis function** set how flexible the model is *before* fitting —
  inspect the prior predictive (Section C.3) to sanity-check that your priors
  produce plausible curves.
- The SHASH **shape parameters are only weakly identified** for strongly skewed
  features, so fits can be sensitive. Prefer informative `epsilon`/`delta`
  priors, enough `tune`/`draws`, and a fixed sampling seed for reproducibility.